In [45]:
import numpy as np 

In [46]:
# measure the time it takes to run a model 
from functools import wraps
from time import time 

def timing(f):
    @wraps(f)
    def wrap(*args,**kw):
        ts= time()
        result = f(*args,**kw)
        te= time()
        print('func:%r args:[%r, %r] took: %2.4f sec' % 
             (f._name_,args,kw,te-ts))
        return result
    return wrap 



In [47]:
#initialize parameters 
S0 =100 #underlying price curr
K=100 #strike price
T=1 #time to maturity in years 
r =0.06 #risk free return rate 
N =3 #number of time steps 
u =1.1 #up factor 
d =1/u
opttype='C' #option type call

In [48]:
def binomial_tree_slow(K,T,S0,r,N,u,d,opttype='C'):
    dt=T/N
    q=(np.exp(r*dt)-d)/(u-d)
    disc = np.exp(-r*dt) #discounting 
    #initialize stock prices at the maturity part of tree
    S=np.zeros(N+1)
    S[0]=S0*d**N
    for j in range(1,N+1):
        S[j]=S[j-1]*u/d
    C = np.zeros(N+1)
    for j in range(0,N+1):
        C[j] = max(0,S[j]-K)
    #step backwards through tree
    for i in np.arange(N,0,-1):
        for j in range(0,i):
            C[j]= disc*(q*C[j+1]+(1-q)*C[j])
    return C[0]
    

In [50]:
binomial_tree_slow(K,T,S0,r,N,u,d,opttype='C')

np.float64(10.145735799928817)

In [57]:
def binomial_tree_fast(K,T,S0,r,N,u,d,opttype='C'):
    dt=T/N
    q=(np.exp(r*dt)-d)/(u-d)
    disc = np.exp(-r*dt) #discounting 
    #initialize stock prices at the maturity part of tree
    C = S0*d**(np.arange(N,-1,-1))*u**(np.arange(0,N+1,1))
    
    #initializeoption values at maturity 
    C = np.maximum(C-K,np.zeros(N+1))

    #step backwards through tree
    for i in np.arange(N,0,-1):
        C = disc*(q*C[1:i+1]+(1-q)*C[0:i])
    return C[0]



In [58]:
binomial_tree_fast(K,T,S0,r,N,u,d,opttype='C')

np.float64(10.145735799928826)